# Setup & Imports

In [1]:
import os
import random
import numpy as np
import torch
import torchaudio
import librosa
from pathlib import Path
from datasets import load_from_disk, Dataset, DatasetDict, Audio
from typing import Optional
from tqdm.auto import tqdm

/home/peter/miniconda3/envs/thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

In [ ]:
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / '.git').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH       = PROJECT_ROOT / 'data' / 'synthetic' / 'paired'
OLD_DATASET         = PROJECT_ROOT / 'data' / 'synthetic' / 'paired_old'
SYNTHETIC_AUDIO_DIR = PROJECT_ROOT / 'data' / 'synthetic' / 'audio'
TEANGLANN_AUDIO_DIR = PROJECT_ROOT / 'data' / 'teanglann' / 'wav_files'
OUTPUT_PATH        = PROJECT_ROOT / 'data' / 'synthetic' / 'l2_training_unpaired'

In [3]:
paired_ds = load_from_disk(str(DATASET_PATH))
print(paired_ds)

Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Dataset({
    features: ['audio', 'phonetic', 'English ASR transcriptions'],
    num_rows: 19136
})


In [4]:
paired_ds[0]

{'audio': {'path': 'carbad.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'k ʌ ɾ ə b ʌ d'}

# Prepare Data

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.data_handling.collapse_phonemes import normalize_row

In [ ]:
paired_ds = paired_ds.map(
    lambda row: normalize_row(row, 'phonetic'),
    load_from_cache_file=False,
    desc="Normalizing phonetic column"
)